In [2]:
import numpy as np
import pandas as pd

In [5]:
df = pd.read_csv(r"../UK-Train-Rides/railway.csv")

In [8]:
df.head()

,Transaction ID,Date of Purchase,Time of Purchase,Purchase Type,Payment Method,Railcard,Ticket Class,Ticket Type,Price,Departure Station,Arrival Destination,Date of Journey,Departure Time,Arrival Time,Actual Arrival Time,Journey Status,Reason for Delay,Refund Request
0,da8a6ba8-b3dc-4677-b176,12/8/2023,12:41:11,Online,Contactless,Adult,Standard,Advance,43,London Paddington,Liverpool Lime Street,1/1/2024,11:00:00,13:30:00,13:30:00,On Time,NaN,No
1,b0cdd1b0-f214-4197-be53,12/16/2023,11:23:01,Station,Credit Card,Adult,Standard,Advance,23,London Kings Cross,York,1/1/2024,9:45:00,11:35:00,11:40:00,Delayed,Signal Failure,No
2,f3ba7a96-f713-40d9-9629,12/19/2023,19:51:27,Online,Credit Card,NaN,Standard,Advance,3,Liverpool Lime Street,Manchester Piccadilly,1/2/2024,18:15:00,18:45:00,18:45:00,On Time,NaN,No
3,b2471f11-4fe7-4c87-8ab4,12/20/2023,23:00:36,Station,Credit Card,NaN,Standard,Advance,13,London Paddington,Reading,1/1/2024,21:30:00,22:30:00,22:30:00,On Time,NaN,No
4,2be00b45-0762-485e-a7a3,12/27/2023,18:22:56,Online,Contactless,NaN,Standard,Advance,76,Liverpool Lime Street,London Euston,1/1/2024,16:45:00,19:00:00,19:00:00,On Time,NaN,No


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31653 entries, 0 to 31652
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Transaction ID       31653 non-null  object
 1   Date of Purchase     31653 non-null  object
 2   Time of Purchase     31653 non-null  object
 3   Purchase Type        31653 non-null  object
 4   Payment Method       31653 non-null  object
 5   Railcard             10735 non-null  object
 6   Ticket Class         31653 non-null  object
 7   Ticket Type          31653 non-null  object
 8   Price                31653 non-null  int64 
 9   Departure Station    31653 non-null  object
 10  Arrival Destination  31653 non-null  object
 11  Date of Journey      31653 non-null  object
 12  Departure Time       31653 non-null  object
 13  Arrival Time         31653 non-null  object
 14  Actual Arrival Time  29773 non-null  object
 15  Journey Status       31653 non-null  object
 16  Reas

In [10]:
df.duplicated().sum()

np.int64(0)

In [11]:
df.isnull().sum()

Transaction ID             0
Date of Purchase           0
Time of Purchase           0
Purchase Type              0
Payment Method             0
Railcard               20918
Ticket Class               0
Ticket Type                0
Price                      0
Departure Station          0
Arrival Destination        0
Date of Journey            0
Departure Time             0
Arrival Time               0
Actual Arrival Time     1880
Journey Status             0
Reason for Delay       27481
Refund Request             0
dtype: int64

In [6]:
df['Railcard'] = df['Railcard'].fillna('No Railcard')
df['Reason for Delay'] = df['Reason for Delay'].fillna('No Delay')


In [13]:
df['Actual Arrival Time'] = df.apply(
    lambda row: row['Arrival Time']
    if pd.isna(row['Actual Arrival Time']) and row['Journey Status'] == 'On Time'
    else row['Actual Arrival Time'],
    axis=1
)

In [14]:
df.dtypes

Transaction ID         object
Date of Purchase       object
Time of Purchase       object
Purchase Type          object
Payment Method         object
Railcard               object
Ticket Class           object
Ticket Type            object
Price                   int64
Departure Station      object
Arrival Destination    object
Date of Journey        object
Departure Time         object
Arrival Time           object
Actual Arrival Time    object
Journey Status         object
Reason for Delay       object
Refund Request         object
dtype: object

In [15]:
df['Date of Purchase'] = pd.to_datetime(df['Date of Purchase'], errors='coerce')
df['Date of Journey'] = pd.to_datetime(df['Date of Journey'], errors='coerce')

In [16]:
time_columns=['Time of Purchase','Departure Time','Actual Arrival Time','Arrival Time']
for col in time_columns:
  df[col] = pd.to_datetime(df[col], format='%H:%M:%S', errors='coerce').dt.time

In [17]:
category_columns = [
    'Purchase Type', 'Payment Method', 'Railcard', 'Ticket Class',
    'Ticket Type', 'Departure Station', 'Arrival Destination',
    'Journey Status', 'Refund Request'
]

for col in category_columns:
    df[col] = df[col].astype('category')

In [18]:
df.dtypes

Transaction ID                 object
Date of Purchase       datetime64[ns]
Time of Purchase               object
Purchase Type                category
Payment Method               category
Railcard                     category
Ticket Class                 category
Ticket Type                  category
Price                           int64
Departure Station            category
Arrival Destination          category
Date of Journey        datetime64[ns]
Departure Time                 object
Arrival Time                   object
Actual Arrival Time            object
Journey Status               category
Reason for Delay               object
Refund Request               category
dtype: object

In [19]:
df['Price'].describe()

count    31653.000000
mean        23.439200
std         29.997628
min          1.000000
25%          5.000000
50%         11.000000
75%         35.000000
max        267.000000
Name: Price, dtype: float64

In [20]:
Q1 = df['Price'].quantile(0.25)
Q3 = df['Price'].quantile(0.75)
IQR = Q3 - Q1
upper_limit = Q3 + 1.5 * IQR
lower_limit = Q1 - 1.5 * IQR


In [21]:
df[(df['Price'] > upper_limit) | (df['Price'] < lower_limit)]

,Transaction ID,Date of Purchase,Time of Purchase,Purchase Type,Payment Method,Railcard,Ticket Class,Ticket Type,Price,Departure Station,Arrival Destination,Date of Journey,Departure Time,Arrival Time,Actual Arrival Time,Journey Status,Reason for Delay,Refund Request
25,842da93c-b820-42dc-ad4f,2023-12-31,15:19:53,Online,Contactless,No Railcard,Standard,Advance,86,Manchester Piccadilly,London Paddington,2024-01-01,13:45:00,16:00:00,16:00:00,On Time,No Delay,No
45,767314a0-f839-4607-a3d3,2024-01-01,05:09:30,Station,Credit Card,No Railcard,First Class,Advance,134,Manchester Piccadilly,London Euston,2024-01-02,03:30:00,05:20:00,05:31:00,Delayed,Weather Conditions,No
51,382d60f9-9fe0-4920-97e4,2024-01-01,06:34:08,Station,Credit Card,No Railcard,Standard,Anytime,151,Liverpool Lime Street,London Euston,2024-01-01,08:00:00,10:15:00,10:39:00,Delayed,Weather,No
61,711c08ba-eb61-44ba-821a,2024-01-01,09:30:09,Station,Credit Card,No Railcard,First Class,Advance,134,Manchester Piccadilly,London Euston,2024-01-02,08:00:00,09:50:00,10:08:00,Delayed,Weather,No
68,9082a416-480e-4ca4-bf9d,2024-01-01,15:39:11,Station,Credit Card,No Railcard,Standard,Anytime,151,Liverpool Lime Street,London Euston,2024-01-01,17:00:00,19:15:00,19:15:00,On Time,No Delay,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31607,606be109-f8e2-4267-b63f,2024-04-30,12:17:40,Online,Credit Card,No Railcard,First Class,Off-Peak,203,Manchester Piccadilly,London Paddington,2024-04-30,13:45:00,16:00:00,16:00:00,On Time,No Delay,No
31626,f71e0949-1e43-4f9b-b026,2024-04-30,15:39:17,Station,Credit Card,No Railcard,Standard,Anytime,151,Liverpool Lime Street,London Euston,2024-04-30,17:00:00,19:15:00,19:15:00,On Time,No Delay,No
31637,5dd5c27f-fd92-42c5-95f9,2024-04-30,17:05:54,Station,Debit Card,Adult,Standard,Anytime,101,Liverpool Lime Street,London Euston,2024-04-30,17:30:00,19:45:00,20:13:00,Delayed,Technical Issue,Yes
31639,465e3643-fb67-4deb-8ec9,2024-04-30,17:13:32,Station,Debit Card,Senior,First Class,Anytime,144,London Euston,Manchester Piccadilly,2024-04-30,18:45:00,20:35:00,NaT,Cancelled,Signal failure,No


In [22]:
text_columns = [
    'Purchase Type', 'Payment Method', 'Railcard', 'Ticket Class',
    'Ticket Type', 'Departure Station', 'Arrival Destination',
    'Journey Status', 'Reason for Delay', 'Refund Request'
]

for col in text_columns:
    df[col] = df[col].astype(str).str.strip().str.lower()

In [23]:
df.head()

,Transaction ID,Date of Purchase,Time of Purchase,Purchase Type,Payment Method,Railcard,Ticket Class,Ticket Type,Price,Departure Station,Arrival Destination,Date of Journey,Departure Time,Arrival Time,Actual Arrival Time,Journey Status,Reason for Delay,Refund Request
0,da8a6ba8-b3dc-4677-b176,2023-12-08,12:41:11,online,contactless,adult,standard,advance,43,london paddington,liverpool lime street,2024-01-01,11:00:00,13:30:00,13:30:00,on time,no delay,no
1,b0cdd1b0-f214-4197-be53,2023-12-16,11:23:01,station,credit card,adult,standard,advance,23,london kings cross,york,2024-01-01,09:45:00,11:35:00,11:40:00,delayed,signal failure,no
2,f3ba7a96-f713-40d9-9629,2023-12-19,19:51:27,online,credit card,no railcard,standard,advance,3,liverpool lime street,manchester piccadilly,2024-01-02,18:15:00,18:45:00,18:45:00,on time,no delay,no
3,b2471f11-4fe7-4c87-8ab4,2023-12-20,23:00:36,station,credit card,no railcard,standard,advance,13,london paddington,reading,2024-01-01,21:30:00,22:30:00,22:30:00,on time,no delay,no
4,2be00b45-0762-485e-a7a3,2023-12-27,18:22:56,online,contactless,no railcard,standard,advance,76,liverpool lime street,london euston,2024-01-01,16:45:00,19:00:00,19:00:00,on time,no delay,no


In [24]:
df.to_csv('cleaned_dataset.csv', index=False)
